In [6]:
#!/usr/bin/env python3
"""
TA-GNN Data Statistics Verification Script
Run: python verify_stats.py
"""

import torch
import pandas as pd
import numpy as np

print("="*70)
print("TA-GNN DATA STATISTICS VERIFICATION")
print("="*70)

# ============================================
# 1. Load the main data tensor (v4)
# ============================================
print("\n[1] Loading data_v4.pt...")
data = torch.load('/home/jovyan/GNN/data/processed/tensors/data_v4.pt', map_location='cpu')

# Inspect structure
print(f"    Data type: {type(data)}")
if isinstance(data, dict):
    print(f"    Keys: {list(data.keys())}")
    for k, v in data.items():
        if isinstance(v, torch.Tensor):
            print(f"      {k}: shape={v.shape}, dtype={v.dtype}")
        elif isinstance(v, np.ndarray):
            print(f"      {k}: shape={v.shape}, dtype={v.dtype}")
        else:
            print(f"      {k}: {type(v)}")

# ============================================
# 2. Extract tensors (adjust keys based on output above)
# ============================================
print("\n[2] Extracting tensors...")

# Common key names - adjust based on actual keys printed above
if isinstance(data, dict):
    # Try common key names
    Y = data.get('Y') or data.get('y') or data.get('demand') or data.get('demand_tensor')
    M = data.get('M') or data.get('m') or data.get('mask') or data.get('operational_mask')
    
    if Y is not None:
        if isinstance(Y, torch.Tensor):
            Y = Y.numpy()
        print(f"    Demand tensor Y: shape={Y.shape}")
    
    if M is not None:
        if isinstance(M, torch.Tensor):
            M = M.numpy()
        print(f"    Mask tensor M: shape={M.shape}")

# ============================================
# 3. Load station metadata
# ============================================
print("\n[3] Loading station metadata...")
stations = pd.read_csv('/home/jovyan/GNN/data/processed/unified/station_metadata_v4.csv')
print(f"    Columns: {list(stations.columns)}")
print(f"    Rows: {len(stations)}")

# ============================================
# 4. Load raw sessions for verification
# ============================================
print("\n[4] Loading raw sessions...")
sessions = pd.read_csv('/home/jovyan/GNN/data/processed/unified/all_sessions.csv')
print(f"    Total raw sessions: {len(sessions):,}")
print(f"    Columns: {list(sessions.columns)[:10]}...")

# ============================================
# 5. Compute statistics
# ============================================
print("\n" + "="*70)
print("COMPUTED STATISTICS")
print("="*70)

if Y is not None and M is not None:
    T, N = Y.shape if len(Y.shape) == 2 else (Y.shape[0], Y.shape[1])
    
    print(f"\n[A] DIMENSIONS")
    print(f"    Stations (N): {N}")
    print(f"    Hourly timestamps (T): {T:,}")
    
    print(f"\n[B] SESSIONS")
    total_sessions = int(Y.sum())
    print(f"    Total sessions (sum of Y): {total_sessions:,}")
    print(f"    Raw sessions (CSV rows): {len(sessions):,}")
    
    print(f"\n[C] OBSERVED STATION-HOURS")
    observed = int(M.sum())
    print(f"    Observed (mask=1): {observed:,}")
    print(f"    Total possible (T×N): {T*N:,}")
    
    print(f"\n[D] DEMAND DISTRIBUTION")
    positive_mask = (Y > 0) & (M == 1)
    positive_hours = int(positive_mask.sum())
    zero_hours = observed - positive_hours
    pct_positive = 100 * positive_hours / observed
    ratio = zero_hours / positive_hours if positive_hours > 0 else float('inf')
    
    print(f"    Positive station-hours (y>0 & m=1): {positive_hours:,} ({pct_positive:.1f}%)")
    print(f"    Zero station-hours (y=0 & m=1): {zero_hours:,} ({100-pct_positive:.1f}%)")
    print(f"    Zero-to-positive ratio: {ratio:.2f}:1")
    
    print(f"\n[E] POSITIVE DEMAND VALUES")
    pos_vals = Y[positive_mask]
    print(f"    Mean: {pos_vals.mean():.2f}")
    print(f"    Median: {np.median(pos_vals):.1f}")
    print(f"    Max: {pos_vals.max():.0f}")

# Stations with coordinates
print(f"\n[F] STATION COORDINATES")
coord_cols = [c for c in stations.columns if 'lat' in c.lower() or 'lon' in c.lower() or 'coord' in c.lower()]
print(f"    Coordinate columns: {coord_cols}")

if 'latitude' in stations.columns:
    has_coords = stations['latitude'].notna().sum()
elif 'lat' in stations.columns:
    has_coords = stations['lat'].notna().sum()
else:
    has_coords = "Check columns above"

if isinstance(has_coords, int):
    pct = 100 * has_coords / len(stations)
    print(f"    Stations with coordinates: {has_coords} ({pct:.1f}%)")

# ============================================
# 6. COMPARISON TABLE
# ============================================
print("\n" + "="*70)
print("COMPARISON: PAPER vs ACTUAL")
print("="*70)
print(f"{'Metric':<35} {'Main Text':<18} {'Appendix':<18} {'ACTUAL':<18}")
print("-"*89)

if Y is not None and M is not None:
    print(f"{'Stations (N)':<35} {'118':<18} {'118':<18} {f'{N}':<18}")
    print(f"{'Hourly timestamps (T)':<35} {'28,758':<18} {'28,758':<18} {f'{T:,}':<18}")
    print(f"{'Total sessions':<35} {'83,247':<18} {'83,078':<18} {f'{total_sessions:,}':<18}")
    print(f"{'Observed station-hours':<35} {'2,174,036':<18} {'1,105,292':<18} {f'{observed:,}':<18}")
    print(f"{'Positive station-hours':<35} {'127,891 (5.9%)':<18} {'63,927 (5.8%)':<18} {f'{positive_hours:,} ({pct_positive:.1f}%)':<18}")
    print(f"{'Zero-to-positive ratio':<35} {'~16:1':<18} {'16.29:1':<18} {f'{ratio:.2f}:1':<18}")

if isinstance(has_coords, int):
    print(f"{'Stations with coordinates':<35} {'89 (75.4%)':<18} {'97 (82.2%)':<18} {f'{has_coords} ({pct:.1f}%)':<18}")

print("\n" + "="*70)
print("ACTION ITEMS")
print("="*70)
print("Update your paper (Section 3 and Appendix B) with the ACTUAL values above.")
print("="*70)

TA-GNN DATA STATISTICS VERIFICATION

[1] Loading data_v4.pt...


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray._reconstruct])` or the `torch.serialization.safe_globals([numpy.core.multiarray._reconstruct])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch

In [2]:
find ~/path/to/your/project -name "*.npy" -o -name "*.pkl" -o -name "*.csv" | head -20


SyntaxError: invalid syntax (2750377964.py, line 1)